# AgentTrace — Quickstart (middleware)

`AgentTraceMiddleware` streams a **deepagents / LangChain `create_agent`** run to an [AgentTrace](https://pypi.org/project/deepagents-trace/) instance as a live sequence diagram (User → Orchestrator → LLM / tools / sub-agents).

Use the middleware when your agent graph is **built per invocation**. For a server that compiles the agent once and reuses it across requests, use the callback handler instead — see `02_callback_handler.ipynb`.

```bash
pip install agenttrace-langchain
```

> This notebook runs **fully offline**: a fake chat model plus a patched HTTP client, so you see the emitted events without a running AgentTrace server or API key. The real-usage config is shown at the end.

In [ ]:
# A fake tool-calling model so the notebook needs no API key / network.
from langchain_core.messages import AIMessage
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.tools import tool


class FakeToolModel(FakeMessagesListChatModel):
    # create_agent binds tools onto the model; the fake ignores them.
    def bind_tools(self, tools, **kwargs):
        return self


@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


# Turn 1: the model calls the `add` tool. Turn 2: it answers.
model = FakeToolModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "add", "args": {"a": 2, "b": 3}, "id": "c1"}]),
    AIMessage(content="The sum is 5."),
])

## Attach the middleware

One `AgentTraceMiddleware` instance = one AgentTrace run. Emission is **non-blocking** (a background thread drains a queue) and **never fatal** (a missing key or the first HTTP error disables tracing for the run and logs one warning — the agent keeps running).

In [ ]:
from unittest.mock import patch
import requests
from agenttrace_langchain import AgentTraceMiddleware
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model,
    tools=[add],
    middleware=[AgentTraceMiddleware(run_name="quickstart", api_key="atr_demo")],
)

# Capture what the middleware POSTs to /api/events (offline).
posted = []


def _fake_post(url, **kwargs):
    posted.append(kwargs.get('json'))
    resp = requests.Response()
    resp.status_code = 201
    resp._content = b'{"runId": "run_demo", "event": {}}'
    return resp


with patch('requests.post', side_effect=_fake_post):
    agent.invoke({"messages": [{"role": "user", "content": "what is 2 + 3?"}]})

for body in posted:
    if 'type' in body:
        print(f"{body['type']:13} {body['source']:13} -> {body['target']:13} {body.get('label','')}")
    elif 'endRun' in body:
        print(f"--- run ended: {body['endRun']} ---")

You should see, in order: `llm_call` → `tool_call` → `tool_result` → `llm_call` → `final_answer`, then the run close.

## Event mapping

| Event | Emitted from | Diagram arrow |
| --- | --- | --- |
| `llm_call` | `wrap_model_call` | Orchestrator → LLM (payload: input + output preview + tokens) |
| `tool_call` | `wrap_tool_call` (before) | Orchestrator → tool |
| `tool_result` | `wrap_tool_call` (after) | tool → Orchestrator |
| `handoff` | tool name matches `handoff`/`delegate`/`task` | Orchestrator → Sub-agent |
| `error` | `wrap_tool_call` raised | tool → Orchestrator (red) |
| `final_answer` | `after_agent` | Orchestrator → User |

## Real usage

Point the middleware at a running AgentTrace instance and a project API key (from the **Integration** tab, prefixed `atr_`). Configure via env vars or explicitly:

| Env var | Default | Purpose |
| --- | --- | --- |
| `AGENTTRACE_URL` | `http://localhost:3000/api/events` | Ingestion endpoint |
| `AGENTTRACE_KEY` | — | Project API key |

```python
from agenttrace_langchain import AgentTraceMiddleware
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model,
    tools=[...],
    middleware=[AgentTraceMiddleware(
        run_name="research run",
        url="https://your-deployment/api/events",  # or set AGENTTRACE_URL
        api_key="atr_...",                          # or set AGENTTRACE_KEY
    )],
)
agent.invoke({"messages": [{"role": "user", "content": "..."}]})
```

Works the same with `await agent.ainvoke(...)` — the middleware's async hooks fire automatically. Launch the dashboard with `pip install deepagents-trace && agenttrace ui`.